# References
leetcode\rectangle_histogram.ipynb
[LeetCode Histogram Rectangle Problem](H:\advent_of_code\leetcode\rectangle_histogram.ipynb)

## LeetCode Maximal Rectangle - Hard

Great question. Let me break down how you'd think through this problem from scratch.

---

## Step 1: Reduce to a simpler problem

The key insight is: **a rectangle in a binary matrix is just a histogram row times some width**.

Look at this matrix row by row:

```
Row 0:  1 0 1 0 0
Row 1:  1 0 1 1 1
Row 2:  1 1 1 1 1
Row 3:  1 0 0 1 0
```

If you build heights as you go down:

```
After row 0:  heights = [1, 0, 1, 0, 0]
After row 1:  heights = [2, 0, 2, 1, 1]
After row 2:  heights = [3, 1, 3, 2, 2]
After row 3:  heights = [4, 0, 0, 3, 0]
```

Now the problem becomes: **at each row, find the largest rectangle in this histogram**. That's the classic "largest rectangle in histogram" problem.

---

## Step 2: What determines a rectangle's width?

For a given column `j` with height `h`, the rectangle that uses this column must:
- Have height **at most** `h` (can't go higher than what's stacked)
- Extend **left** until we hit a column with height < `h`
- Extend **right** until we hit a column with height < `h`

So for each column, you need to know:
- **Left boundary**: the closest column to the left with height < current height
- **Right boundary**: the closest column to the right with height < current height

This is where the two passes come from.

---

## Step 3: Why two passes (left and right)?

You need **two pieces of information** per column: where can it extend left, and where can it extend right.

- **Left boundary**: found by scanning **left-to-right**. As you scan, you track where the current contiguous segment of 1s started. If you hit a 0, the next potential start is after it. If you hit a 1, the left boundary can be no farther left than where this row's current run started AND no farther left than the row above allowed you to go (that's the `max(left[j], curr_left)` part).

- **Right boundary**: found by scanning **right-to-left**. Same logic but in reverse. You track where the current segment of 1s ends. On a 0, the boundary moves left to this column. On a 1, the right boundary can be no farther right than where the current row's run allows AND no farther right than the row above allowed (that's the `min(right[j], curr_right)` part).

You cannot do both in a single pass because:
- When you're at column `j`, you don't yet know what's to your right (for the right boundary).
- When you're scanning right-to-left, you don't know what was to your left (that's already been processed).

---

## Step 4: The three arrays and what they track

The algorithm maintains three arrays across rows:

| Array | Tracks | Initial value | Updated how? |
|-------|--------|---------------|--------------|
| `height[j]` | How many consecutive 1s above column j | 0 | Increment on "1", reset to 0 on "0" |
| `left[j]` | Leftmost column where the rectangle at j can extend | 0 | `max(previous_left[j], curr_left)` — take the more restrictive |
| `right[j]` | Rightmost column + 1 where the rectangle at j can extend | n | `min(previous_right[j], curr_right)` — take the more restrictive |

The `previous_left[j]` comes from the row above — this is how rectangles span multiple rows. Each new row can only further *restrict* the boundaries (never expand them), which is why left uses `max()` (rightmost of the two left boundaries) and right uses `min()` (leftmost of the two right boundaries).

---

## Step 5: Visual walkthrough

Take the matrix after row 2 (heights = `[3, 1, 3, 2, 2]`):

```
Col:   0  1  2  3  4
h =   [3, 1, 3, 2, 2]
```

**Left pass**: track `curr_left`
- j=0, val="1": left[0] = max(prev_left[0]=0, curr_left=0) = 0
- j=1, val="1": left[1] = max(prev_left[1]=0, curr_left=0) = 0
- j=2, val="1": left[2] = max(prev_left[2]=0, curr_left=0) = 0
- j=3, val="1": left[3] = max(prev_left[3]=0, curr_left=0) = 0
- j=4, val="1": left[4] = max(prev_left[4]=0, curr_left=0) = 0

Wait, no zeros in this row! So `curr_left` never updates → left boundaries stay from row above.

Let me use row 1 instead (heights = `[2, 0, 2, 1, 1]`):

```
Col:   0  1  2  3  4
h =   [2, 0, 2, 1, 1]
```

**Left pass**:
- curr_left = 0
- j=0, val="1": left[0] = max(?, 0) — let's say prev_left was [0,0,0,0,0]
  → left[0] = max(0, 0) = 0
- j=1, val="0": height[1] = 0, left[1] = 0, curr_left = 2
- j=2, val="1": left[2] = max(0, 2) = 2 (can't extend past column 1's zero!)
- j=3, val="1": left[3] = max(0, 2) = 2
- j=4, val="1": left[4] = max(0, 2) = 2

**Right pass**:
- curr_right = 5
- j=4, val="1": right[4] = min(prev_right[4]=5, 5) = 5
- j=3, val="1": right[3] = min(prev_right[3]=5, 5) = 5
- j=2, val="1": right[2] = min(prev_right[2]=5, 5) = 5
- j=1, val="0": right[1] = 5, curr_right = 1
- j=0, val="1": right[0] = min(prev_right[0]=5, 1) = 1 (can't extend past column 1!)

So the rectangle at column 0 has height 2, left=0, right=1 → area = 2×(1-0) = 2.
The rectangle at column 2 has height 2, left=2, right=5 → area = 2×(5-2) = 6. That's the maximal rectangle (the 2×3 block).

---

## Summary of the reasoning chain

1. **Reduce**: Largest rectangle in binary matrix → largest rectangle in histogram at each row.
2. **Realize**: A rectangle's width is bounded by the nearest column with smaller height on each side.
3. **Compute**: Need left boundary and right boundary for each column.
4. **Store**: `height` (runs vertically), `left` and `right` (boundaries inherited/capped across rows).
5. **Passes**: Left-to-right for `left` boundaries, right-to-left for `right` boundaries.
6. **Merge**: `area = height[j] × (right[j] - left[j])` at each cell, track max.

Does this breakdown make sense? If so, toggle to Act mode and I'll code it into your notebook.

In [6]:
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        if not matrix or not matrix[0]:
            return 0

        m = len(matrix)        # number of rows
        n = len(matrix[0])     # number of columns

        height = [0] * n
        left   = [0] * n
        right  = [n] * n       # initialize all right boundaries to n

        max_area = 0

        for i in range(m):
            # --- left-to-right pass: update height and left ---
            curr_left = 0
            for j in range(n):
                if matrix[i][j] == "1":
                    height[j] += 1
                    left[j] = max(left[j], curr_left)
                else:
                    height[j] = 0
                    left[j] = 0
                    curr_left = j + 1   # next possible start is after this zero

            # --- right-to-left pass: update right ---
            curr_right = n
            for j in range(n - 1, -1, -1):
                if matrix[i][j] == "1":
                    right[j] = min(right[j], curr_right)
                    area = height[j] * (right[j] - left[j])
                    max_area = max(max_area, area)
                else:
                    right[j] = n
                    curr_right = j      # the zero at j becomes the new boundary

        return max_area

        # Calculate the maximum area of rectangle in a histogram
        # we calculate the height and area of each row and keep 
        # track of the maximum area found so far.


if __name__ == "__main__":
    solution = Solution()
    matrix = [
        ["1", "0", "1", "0", "0"],
        ["1", "0", "1", "1", "1"],
        ["1", "1", "1", "1", "1"],
        ["1", "0", "0", "1", "0"]
    ]
    result = solution.maximalRectangle(matrix)
    print(result)  # Output: 6

0
['1', '0', '1', '0', '0']


TypeError: list indices must be integers or slices, not list

In [ ]:
for row in matrix:
    # left-to-right
    curr_left = 0
    for j, val in enumerate(row):       # j = column index, val = "1" or "0"
        if val == "1":
            height[j] += 1
            left[j] = max(left[j], curr_left)
        else:
            height[j] = 0
            left[j] = 0
            curr_left = j + 1

    # right-to-left
    curr_right = len(row)
    for j in range(len(row) - 1, -1, -1):
        if row[j] == "1":
            right[j] = min(right[j], curr_right)
        else:
            right[j] = len(row)
            curr_right = j


## Histogram Rectangle Problem

I can help answer this — you're referring to how to track contiguous segments of `1`s in the maximal rectangle problem, specifically where runs of `1`s start and end.

In the histogram-based DP approach for maximal rectangle, for each row you track three arrays as you scan column by column:

**1. `height[j]`** — counts consecutive `1`s above (including current row). Increment if `matrix[i][j] == "1"`, reset to 0 otherwise.

**2. `left[j]`** — the leftmost boundary (starting column index) where the current height at column `j` can extend. To track the *start* of a run of 1s:
   - As you scan left-to-right across a row, maintain a `curr_left` variable.
   - When you hit a `"1"`, `curr_left` stays at the first column of the current run (it's the start of the 1s segment).
   - When you hit a `"0"`, reset `curr_left` to `0` and also reset `left[j] = 0` for that column.
   - At each `"1"` cell: `left[j] = max(left[j], curr_left)` — this takes the *most restrictive* (rightmost) left boundary between the current row and the row above.

**3. `right[j]`** — the rightmost boundary + 1 where the height extends. To track the *end* of a run of 1s:
   - Scan right-to-left, maintain a `curr_right` variable starting at `n` (number of columns).
   - When you hit a `"1"`, `curr_right` stays where it is (tracking the position just after the current run ends).
   - When you hit a `"0"`, reset `curr_right = j` (the end boundary is this zero's index).
   - At each `"1"` cell: `right[j] = min(right[j], curr_right)` — this takes the *most restrictive* (leftmost) right boundary.

**4. Area** at each cell `(i, j)` = `height[j] * (right[j] - left[j])`.

In other words:
- **Start of the 1s**: tracked by `curr_left` — it marks the column index where the current contiguous segment of 1s in this row began.
- **End of the 1s**: tracked by `curr_right` (in the reverse pass) — it marks the column index *after* where the current segment of 1s ends.

If you'd like, I can help you implement this directly in your notebook by filling in the `maximalRectangle` method. Would you like me to provide the code, or do you want to toggle to Act mode so I can write it into the notebook?

curr_right starts at n (the number of columns).

Here's why:

You scan right-to-left across a row. The variable curr_right represents the column index just after the current contiguous segment of 1s ends. Since you're scanning from the right edge moving left:

Initial value: curr_right = n — because before you've seen any 1s, the "end" boundary is conceptually past the last column.
When you hit a "0" at column j: set curr_right = j — this marks that any 1s to the left of this zero will have their right boundary at this column (the zero itself).
When you hit a "1" at column j: right[j] = min(right[j], curr_right) — the right boundary for this cell is whatever curr_right currently is (the nearest zero to its right, or n if there is none).
So for the example matrix:


["1", "0", "1", "1", "1"]   (row 1, 0-indexed)
  0     1    2    3    4
On row 1 (second row), when processing for the first time and scanning right-to-left, curr_right would behave like:

Start: curr_right = 5 (n = 5 columns)
j=4 ("1"): right[4] = 5 (no zero to its right), curr_right stays 5
j=3 ("1"): right[3] = 5
j=2 ("1"): right[2] = 5
j=1 ("0"): curr_right = 1 (reset)
j=0 ("1"): right[0] = 1 (the nearest zero is at column 1)
So curr_right starts at n and only gets smaller when you encounter a "0", effectively tracking the "end" boundary (exclusive) of the current run of 1s.




Yes — `curr_right` starts at `n` (the number of columns).

The reason: you're scanning right-to-left, and `curr_right` tracks the column index *just after* where the current run of 1s ends. Before you've seen any zeros, the "end" is conceptually past the last column, so `n` is the starting value.

Here's how it behaves on a row like `["1", "0", "1", "1", "1"]` (n=5):

| j | value | curr_right before | action | curr_right after | right[j] |
|---|-------|-------------------|--------|------------------|----------|
| 4 | "1"   | 5 (initial)      | no zero hit, keep curr_right | 5 | min(right[4], 5) |
| 3 | "1"   | 5                 | keep | 5 | min(right[3], 5) |
| 2 | "1"   | 5                 | keep | 5 | min(right[2], 5) |
| 1 | "0"   | 5                 | hit zero → curr_right = j | **1** | right[1] = 0 (gets reset) |
| 0 | "1"   | 1                 | keep | 1 | min(right[0], 1) — boundary at column 1 |

So `curr_right` acts as a right boundary pointer that only moves left when a zero is encountered. Would you like me to walk through a full example with the code, or are you ready to implement it?

Absolutely. Let me start with the histogram problem, then build up to the full maximal rectangle code line-by-line.

---

## Part 1: Largest Rectangle in Histogram

This is the foundation. A histogram is just an array of bar heights:

```
heights = [2, 1, 5, 6, 2, 3]
```

Visualized:

```
    ██
    ██
    ██  ██
    ██  ██  ██
██  ██  ██  ██  ██
██  ██  ██  ██  ██  ██
2   1   5   6   2   3
```

**Goal**: Find the largest rectangle that can be formed entirely within the bars.

### Key insight

For each bar of height `h`, the largest rectangle that **includes that bar** has:
- Height = `h` (can't be taller than the bar itself)
- Width = extends left until a bar shorter than `h`, extends right until a bar shorter than `h`

So for each bar, you need to find the **nearest smaller bar to the left** and the **nearest smaller bar to the right**.

### Example: bar at index 2 (height = 5)

```
heights = [2, 1, 5, 6, 2, 3]
           0  1  2  3  4  5
```

- Nearest smaller to the left: index 1 (height 1)
- Nearest smaller to the right: index 4 (height 2)
- Width = 4 - 1 - 1 = 2 (columns 2 and 3)
- Area = 5 × 2 = 10

### Example: bar at index 3 (height = 6)

- Nearest smaller to the left: index 1 (height 1) — wait, index 2 has height 5 which is smaller than 6, so index 2
- Nearest smaller to the right: index 4 (height 2)
- Width = 4 - 2 - 1 = 1 (just column 3)
- Area = 6 × 1 = 6

### The stack-based solution (optimal)

You use a **monotonic stack** (increasing heights) to find left and right boundaries in one pass:

```python
def largestRectangleArea(heights):
    stack = []           # stores indices, heights are increasing
    max_area = 0
    n = len(heights)
    
    for i, h in enumerate(heights + [0]):  # append 0 to force cleanup
        while stack and heights[stack[-1]] > h:
            # The bar at stack[-1] just found its right boundary at i
            height = heights[stack.pop()]
            # Left boundary is the new top of stack (or -1 if empty)
            left = stack[-1] if stack else -1
            width = i - left - 1
            max_area = max(max_area, height * width)
        stack.append(i)
    
    return max_area
```

But the maximal rectangle problem uses a **different approach** (the left/right array method) because it needs to **carry boundaries across rows**. The stack method resets each row, which doesn't work when you need to track how boundaries change row by row.

---

## Part 2: The maximal rectangle approach (left/right arrays)

Instead of recomputing from scratch each row, we maintain `left[j]` and `right[j]` arrays that get **restricted** (narrowed) as we go down rows. This is why we use the two-pass method rather than a stack.

---

## Part 3: Full code line-by-line with comments

```python
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        # --- GUARD CLAUSE ---
        # If the matrix is empty, there are no rectangles
        if not matrix or not matrix[0]:
            return 0

        # --- DIMENSIONS ---
        m = len(matrix)          # number of rows
        n = len(matrix[0])       # number of columns

        # --- THREE ARRAYS (one per column) ---
        # height[j]: consecutive 1s above column j (including current row)
        # left[j]:   leftmost column where the rectangle at j can extend
        # right[j]:  rightmost column + 1 where the rectangle at j can extend
        height = [0] * n
        left   = [0] * n
        right  = [n] * n         # initially, no restriction — boundary is past the last column

        max_area = 0

        # --- PROCESS EACH ROW ---
        for i in range(m):       # i = row index
            row = matrix[i]      # current row as a list of strings

            # ==========================================
            # PASS 1: LEFT-TO-RIGHT
            # Update height[j] and left[j]
            # ==========================================
            curr_left = 0        # tracks where the current run of 1s started in THIS row
                                 # resets to 0 at the start of each row

            for j in range(n):   # j = column index, scanning left to right
                if row[j] == "1":
                    # --- HEIGHT ---
                    # If this cell is "1", the column's height grows by 1
                    # (it's stacked on top of previous rows' 1s)
                    height[j] += 1

                    # --- LEFT BOUNDARY ---
                    # left[j] from the row above is the most restrictive left boundary
                    # from previous rows. curr_left is the left boundary from THIS row.
                    # We take the MAX (rightmost) because the rectangle can't extend
                    # past the more restrictive (closer) boundary.
                    # Example: if row above says left=0 but this row has a zero at col 1,
                    # curr_left=2, so left[j] = max(0, 2) = 2
                    left[j] = max(left[j], curr_left)

                else:  # row[j] == "0"
                    # --- HEIGHT ---
                    # A zero breaks the vertical stack — height resets to 0
                    height[j] = 0

                    # --- LEFT BOUNDARY ---
                    # Reset left[j] to 0 (no restriction from above, since height is 0)
                    left[j] = 0

                    # --- CURR_LEFT ---
                    # The current run of 1s ends at this zero.
                    # The next potential run starts at column j+1
                    curr_left = j + 1

            # ==========================================
            # PASS 2: RIGHT-TO-LEFT
            # Update right[j] and calculate area
            # ==========================================
            curr_right = n       # tracks where the current run of 1s ENDS (exclusive)
                                 # starts at n (past the last column)
                                 # resets to n at the start of each row's right pass

            for j in range(n - 1, -1, -1):  # j = n-1, n-2, ..., 0
                if row[j] == "1":
                    # --- RIGHT BOUNDARY ---
                    # right[j] from the row above is the most restrictive right boundary
                    # from previous rows. curr_right is the right boundary from THIS row.
                    # We take the MIN (leftmost) because the rectangle can't extend
                    # past the more restrictive (closer) boundary.
                    right[j] = min(right[j], curr_right)

                    # --- CALCULATE AREA ---
                    # At this point, we know:
                    #   height[j] = how tall the stack is at column j
                    #   left[j]   = leftmost column the rectangle can extend to
                    #   right[j]  = rightmost column + 1 the rectangle can extend to
                    # Width = right[j] - left[j]
                    # Area = height[j] * width
                    area = height[j] * (right[j] - left[j])
                    max_area = max(max_area, area)

                else:  # row[j] == "0"
                    # --- RIGHT BOUNDARY ---
                    # Reset right[j] to n (no restriction, since height is 0)
                    right[j] = n

                    # --- CURR_RIGHT ---
                    # The current run of 1s ends at this zero.
                    # The boundary moves to this column (j).
                    # Any 1s to the left of this zero will have their right boundary
                    # at most j (they can't extend past this zero).
                    curr_right = j

        return max_area
```

---

## Part 4: Trace through the example

Matrix:
```
Row 0:  1 0 1 0 0
Row 1:  1 0 1 1 1
Row 2:  1 1 1 1 1
Row 3:  1 0 0 1 0
```

### Row 0: `[1, 0, 1, 0, 0]`

**Before row 0**: height=[0,0,0,0,0], left=[0,0,0,0,0], right=[5,5,5,5,5]

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=1, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="1": height[2]=1, left[2]=max(0,2)=2
- j=3, val="0": height[3]=0, left[3]=0, curr_left=4
- j=4, val="0": height[4]=0, left[4]=0, curr_left=5

**Right pass**:
- curr_right=5
- j=4, val="0": right[4]=5, curr_right=4
- j=3, val="0": right[3]=5, curr_right=3
- j=2, val="1": right[2]=min(5,3)=3, area=1×(3-2)=1, max=1
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(5,1)=1, area=1×(1-0)=1, max=1

After row 0: height=[1,0,1,0,0], left=[0,0,2,0,0], right=[1,5,3,5,5]

### Row 1: `[1, 0, 1, 1, 1]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=2, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="1": height[2]=2, left[2]=max(2,2)=2
- j=3, val="1": height[3]=1, left[3]=max(0,2)=2
- j=4, val="1": height[4]=1, left[4]=max(0,2)=2

**Right pass**:
- curr_right=5
- j=4, val="1": right[4]=min(5,5)=5, area=1×(5-2)=3, max=3
- j=3, val="1": right[3]=min(5,5)=5, area=1×(5-2)=3, max=3
- j=2, val="1": right[2]=min(3,5)=3, area=2×(3-2)=2, max=3
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(1,1)=1, area=2×(1-0)=2, max=3

After row 1: height=[2,0,2,1,1], left=[0,0,2,2,2], right=[1,5,3,5,5]

### Row 2: `[1, 1, 1, 1, 1]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=3, left[0]=max(0,0)=0
- j=1, val="1": height[1]=1, left[1]=max(0,0)=0
- j=2, val="1": height[2]=3, left[2]=max(2,0)=2
- j=3, val="1": height[3]=2, left[3]=max(2,0)=2
- j=4, val="1": height[4]=2, left[4]=max(2,0)=2

**Right pass**:
- curr_right=5
- j=4, val="1": right[4]=min(5,5)=5, area=2×(5-2)=6, max=6
- j=3, val="1": right[3]=min(5,5)=5, area=2×(5-2)=6, max=6
- j=2, val="1": right[2]=min(3,5)=3, area=3×(3-2)=3, max=6
- j=1, val="1": right[1]=min(5,5)=5, area=1×(5-0)=5, max=6
- j=0, val="1": right[0]=min(1,5)=1, area=3×(1-0)=3, max=6

After row 2: height=[3,1,3,2,2], left=[0,0,2,2,2], right=[1,5,3,5,5]

### Row 3: `[1, 0, 0, 1, 0]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=4, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="0": height[2]=0, left[2]=0, curr_left=3
- j=3, val="1": height[3]=3, left[3]=max(2,3)=3
- j=4, val="0": height[4]=0, left[4]=0, curr_left=5

**Right pass**:
- curr_right=5
- j=4, val="0": right[4]=5, curr_right=4
- j=3, val="1": right[3]=min(5,4)=4, area=3×(4-3)=3, max=6
- j=2, val="0": right[2]=5, curr_right=2
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(1,1)=1, area=4×(1-0)=4, max=6

**Final answer: 6** ✓

---

## Summary of the key ideas

1. **Histogram reduction**: Each row is a histogram of heights. The maximal rectangle in the matrix is the maximal rectangle in one of these histograms.

2. **Left/right boundaries**: For each column, the rectangle that uses that column is bounded by the nearest column with smaller height on each side.

3. **Two passes**: Left-to-right finds left boundaries (need to know what came before). Right-to-left finds right boundaries (need to know what comes after).

4. **Carrying across rows**: `left[j]` and `right[j]` persist across rows, getting more restrictive (narrower) as we go down. This is the key difference from the stack-based histogram solution.

5. **Area formula**: `height[j] × (right[j] - left[j])` at each cell, track the maximum.

Would you like me to write this into your notebook? If so, toggle to Act mode.